# Explore knowledge base

In [1]:
import lancedb

vector_db = lancedb.connect(uri = "knowledge_base")
vector_db.list_tables().tables

['articles']

In [2]:
vector_db["articles"]

LanceTable(name='articles', _conn=LanceDBConnection(uri='c:\\Users\\anjag\\Documents\\github\\AIOps\\AIOps_course\\code_along\\video\\11_rag\\knowledge_base'))

In [3]:
vector_db["articles"].to_pandas()

,doc_id,filepath,filename,content,embedding
0,dwarf_hotot,C:\Users\anjag\Documents\github\AIOps\AIOps_co...,dwarf_hotot,Dwarf Hotot\nCountry\nof origin\nFrance\nUse S...,"[0.096375, 0.014977, -0.050629, 0.001607, -0.0..."
1,dwarf_rabbit,C:\Users\anjag\Documents\github\AIOps\AIOps_co...,dwarf_rabbit,A dwarf rabbit\n(Netherland Dwarf breed)\nBaby...,"[0.102966, 0.007874, -0.049408, 0.005562, -0.0..."
2,holland_lop,C:\Users\anjag\Documents\github\AIOps\AIOps_co...,holland_lop,Holland Lop\nHolland lop standing on her back ...,"[0.080872, 0.01165, -0.039276, -0.000738, -0.0..."
3,netherland_dwarf_rabbit,C:\Users\anjag\Documents\github\AIOps\AIOps_co...,netherland_dwarf_rabbit,Netherland Dwarf rabbit\nCountry of\norigin\n ...,"[0.07373, 0.021042, -0.049042, 0.014122, -0.01..."


In [4]:
import time
from openai import OpenAI
import os

test_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API"),
    max_retries=1,
    timeout=60.0,
)

t0 = time.perf_counter()
resp = test_client.embeddings.create(
    model="nvidia/nemotron-3.5-embed-1b",
    input=["a short rabbit"],
    extra_body={"input_type": "query"},
)
dt = time.perf_counter() - t0

vec = resp.data[0].embedding
print(f"Klart på {dt:.2f}s | Vektorlängd: {len(vec)}")

Klart på 0.31s | Vektorlängd: 2048


In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API"),
    max_retries=0,
    timeout=15.0,
)

models = [m.id for m in client.models.list().data]
embed_models = [m for m in models if "embed" in m.lower()]
print("Tillgängliga embed-modeller:", embed_models)

Tillgängliga embed-modeller: ['nvidia/embed-qa-4', 'nvidia/llama-3.2-nemoretriever-1b-vlm-embed-v1', 'nvidia/llama-3.2-nv-embedqa-1b-v1', 'nvidia/nemotron-3-embed-1b', 'nvidia/nemotron-3.5-embed-1b', 'nvidia/nemotron-3.5-embed-8b', 'nvidia/nv-embedqa-mistral-7b-v2', 'snowflake/arctic-embed-l']


In [6]:
import lancedb
from backend import data_models  # säkerställer att "nvidia-nemotron" är registrerad

vector_db = lancedb.connect(uri="knowledge_base")
results = vector_db["articles"].search("a short rabbit").limit(3).to_list()
print("Antal träffar:", len(results))

Antal träffar: 3


In [11]:
vector_db["articles"].search("a short rabbit").to_list()[0]["content"]

'A dwarf rabbit\n(Netherland Dwarf breed)\nBaby Mini Satin rabbits\nBlue-eyed white - Satin coat\nDwarf rabbit\nDwarf rabbit refers either (formally) to a rabbit with the\ndwarfing gene,[1] or (informally) to any small breed of\ndomestic rabbit or specimen thereof, or (colloquially) to\nany small rabbit. Dwarfism is a genetic condition that\nmay occur in humans and in many animals, including\nrabbits. True dwarfism is often associated with a cluster\nof physical abnormalities,[2] including pituitary\ndwarfism. The process of dwarfing is used to selectively\nbreed for smaller stature with each generation. Small\nstature is a characteristic of neoteny, which may account\n(in part) for the attraction of dwarf animals.\nThe Netherland Dwarf is the smallest of the domestic rabbits.\nThe American Rabbit Breeders Association (ARBA)[3][4]\naccepts a weight range of 1.1–3.5 lb (0.50–1.59 kg), but 2.5 lb\n(1.1 kg) is the maximum allowed by the British Rabbit\nCouncil (BRC).[5] The small stature 